# DFU Repair-7 — GOOD38 Recovery + CPU Repair

Upload `DFU_GOOD38_RECOVERY_EVIDENCE.zip` when prompted. The notebook reconstructs only missing/invalid GOOD38 per-trial evidence without training, verifies all 38 against the existing locked split, then launches the CPU Repair-7 runner for only the original seven incompatible Fold-1 trials.

In [ ]:
# DFU Repair-7 — GOOD38 upload recovery + CPU repair
import os, io, json, time, shutil, hashlib, zipfile, ast, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

RUN_ID = "RELIABLE_DFU_CV_V3_MISSING38"
RUN_ROOT = Path("/content/drive/MyDrive/DFU-ImageGuard/runs") / RUN_ID
LOCKED_SPLIT = RUN_ROOT / "manifests" / "locked_outer_fold_assignments.csv"
EXPECTED_EVIDENCE_SHA256 = "6b0c149456f5b29a4a132534d0a6e71ac5fb31550d2cc4ef056131219e62ea5b"
EVIDENCE_NAME = "DFU_GOOD38_RECOVERY_EVIDENCE.zip"
FINAL_FIXED_URL = (
    "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/"
    "177ab6fb7311f5e864145ec4f2ea3e156fa05db6/"
    "notebooks/DFU_Repair7_CPU_FINAL_FIXED.ipynb"
)

MODELS = ("convnextv2_tiny", "mobilenetv3_large", "densenet121")
SEEDS = (2026, 2027, 2028)
BAD7 = {
    ("convnextv2_tiny", 2026, 1),
    ("convnextv2_tiny", 2027, 1),
    ("convnextv2_tiny", 2028, 1),
    ("densenet121", 2026, 1),
    ("densenet121", 2027, 1),
    ("mobilenetv3_large", 2026, 1),
    ("mobilenetv3_large", 2027, 1),
}
GOOD38 = sorted({(m, s, f) for m in MODELS for s in SEEDS for f in range(1, 6)} - BAD7)
if len(GOOD38) != 38:
    raise RuntimeError("Protocol identity count error")

def sha256_file(path, chunk=4 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)

def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

def trial_path(model, seed, fold_one):
    return RUN_ROOT / "trials" / model / f"seed_{seed}" / f"fold_{fold_one}"

print("=" * 78)
print("DFU GOOD38 RECOVERY + REPAIR-7 CPU")
print("GOOD38 training: FORBIDDEN")
print("Authorized training: original BAD7 only")
print("=" * 78)

from google.colab import drive, files
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
print("Google Drive mount: PASS")

if not RUN_ROOT.is_dir():
    raise RuntimeError(f"Existing run root not found: {RUN_ROOT}")
if not LOCKED_SPLIT.is_file():
    raise RuntimeError(f"Locked split not found: {LOCKED_SPLIT}")

locked = pd.read_csv(LOCKED_SPLIT)
required = {"image_id", "group_id", "label", "relative_path", "outer_fold"}
if not required.issubset(locked.columns):
    raise RuntimeError(f"Locked split missing columns: {required - set(locked.columns)}")
locked["image_id"] = locked["image_id"].astype(str)
if len(locked) != 1055 or locked["image_id"].duplicated().any():
    raise RuntimeError(
        f"Locked split expected 1055 unique images; rows={len(locked)}, "
        f"unique={locked.image_id.nunique()}"
    )
expected_by_fold = {
    f: set(locked.loc[locked.outer_fold.astype(int) == f, "image_id"])
    for f in range(5)
}
fold_sizes = {f + 1: len(v) for f, v in expected_by_fold.items()}
print("Locked fold sizes:", fold_sizes)
if fold_sizes != {1: 209, 2: 206, 3: 217, 4: 214, 5: 209}:
    raise RuntimeError(f"Unexpected locked fold sizes: {fold_sizes}")

evidence_path = Path("/content") / EVIDENCE_NAME
if not evidence_path.is_file():
    print(f"\nUpload {EVIDENCE_NAME} when the file chooser opens.")
    uploaded = files.upload()
    if EVIDENCE_NAME not in uploaded:
        raise RuntimeError(f"Required evidence file not uploaded: {EVIDENCE_NAME}")
    evidence_path.write_bytes(uploaded[EVIDENCE_NAME])

got_sha = sha256_file(evidence_path)
if got_sha != EXPECTED_EVIDENCE_SHA256:
    raise RuntimeError(
        f"Evidence ZIP SHA mismatch: {got_sha} != {EXPECTED_EVIDENCE_SHA256}"
    )
print("GOOD38 evidence ZIP SHA: PASS")

with zipfile.ZipFile(evidence_path) as z:
    manifest = json.loads(z.read("MANIFEST.json").decode("utf-8"))
    pred_bytes = z.read("good38_predictions.csv")
    metric_bytes = z.read("good38_metrics.csv")

if hashlib.sha256(pred_bytes).hexdigest() != manifest["predictions_sha256"]:
    raise RuntimeError("GOOD38 predictions SHA mismatch")
if hashlib.sha256(metric_bytes).hexdigest() != manifest["metrics_sha256"]:
    raise RuntimeError("GOOD38 metrics SHA mismatch")

evidence_pred = pd.read_csv(io.BytesIO(pred_bytes))
evidence_metric = pd.read_csv(io.BytesIO(metric_bytes))
if len(evidence_pred) != 8032 or len(evidence_metric) != 38:
    raise RuntimeError(
        f"GOOD38 evidence size mismatch: pred={len(evidence_pred)}, "
        f"metrics={len(evidence_metric)}"
    )
print(
    f"Pinned GOOD38 evidence: PASS | identities=38, "
    f"predictions={len(evidence_pred)}, metrics={len(evidence_metric)}"
)

def validate_good(model, seed, fold_one):
    t = trial_path(model, seed, fold_one)
    cp = t / "COMPLETE.json"
    pp = t / "test_predictions.csv"
    if not (cp.is_file() and pp.is_file()):
        return {"valid": False, "reason": "missing_complete_or_predictions", "trial": str(t)}
    try:
        complete = json.loads(cp.read_text(encoding="utf-8"))
        p = pd.read_csv(pp)
        if (
            str(complete.get("model_key")) != model
            or int(complete.get("seed")) != seed
            or int(complete.get("outer_fold")) != fold_one
        ):
            return {"valid": False, "reason": "complete_identity_mismatch", "trial": str(t)}
        need = {
            "image_id", "model_key", "seed", "outer_fold",
            "group_id", "label", "prob_calibrated", "pred",
        }
        if not need.issubset(p.columns):
            return {"valid": False, "reason": "prediction_columns_missing", "trial": str(t)}
        p["image_id"] = p["image_id"].astype(str)
        exp = expected_by_fold[fold_one - 1]
        got = set(p["image_id"])
        if len(p) != len(exp) or p["image_id"].duplicated().any() or got != exp:
            return {
                "valid": False, "reason": "locked_split_mismatch",
                "rows": len(p), "unique": len(got), "expected": len(exp),
                "unexpected": len(got - exp), "missing": len(exp - got),
                "trial": str(t),
            }
        ids = p[["model_key", "seed", "outer_fold"]].drop_duplicates()
        if len(ids) != 1:
            return {"valid": False, "reason": "prediction_identity_not_unique", "trial": str(t)}
        r = ids.iloc[0]
        if str(r.model_key) != model or int(r.seed) != seed or int(r.outer_fold) != fold_one:
            return {"valid": False, "reason": "prediction_identity_mismatch", "trial": str(t)}
        return {"valid": True, "trial": str(t)}
    except Exception as e:
        return {"valid": False, "reason": f"exception:{type(e).__name__}:{e}", "trial": str(t)}

locked_sha = sha256_file(LOCKED_SPLIT)
already_valid, recovered = [], []

for model, seed, fold_one in GOOD38:
    chk = validate_good(model, seed, fold_one)
    if chk.get("valid"):
        already_valid.append((model, seed, fold_one))
        continue

    ep = evidence_pred[
        (evidence_pred.model_key.astype(str) == model)
        & (evidence_pred.seed.astype(int) == seed)
        & (evidence_pred.outer_fold.astype(int) == fold_one)
    ].copy()
    em = evidence_metric[
        (evidence_metric.model_key.astype(str) == model)
        & (evidence_metric.seed.astype(int) == seed)
        & (evidence_metric.outer_fold.astype(int) == fold_one)
    ].copy()

    exp = expected_by_fold[fold_one - 1]
    ep["image_id"] = ep["image_id"].astype(str)
    got = set(ep["image_id"])
    if len(ep) != len(exp) or ep["image_id"].duplicated().any() or got != exp:
        raise RuntimeError(
            f"V4 evidence/locked split mismatch for {(model, seed, fold_one)}: "
            f"rows={len(ep)}, unique={len(got)}, expected={len(exp)}, "
            f"unexpected={len(got-exp)}, missing={len(exp-got)}"
        )
    if len(em) != 1:
        raise RuntimeError(
            f"Expected one metric row for {(model, seed, fold_one)}, found {len(em)}"
        )

    t = trial_path(model, seed, fold_one)
    t.mkdir(parents=True, exist_ok=True)
    backup = RUN_ROOT / "_good38_recovery_backup" / model / f"seed_{seed}" / f"fold_{fold_one}"
    for name in ("COMPLETE.json", "test_predictions.csv", "TRIAL_VERIFICATION.json"):
        old = t / name
        if old.exists():
            backup.mkdir(parents=True, exist_ok=True)
            shutil.copy2(old, backup / f"{name}.pre_recovery_{time.time_ns()}")

    metric_record = {}
    for k, v in em.iloc[0].to_dict().items():
        if pd.isna(v):
            metric_record[k] = None
        elif isinstance(v, np.generic):
            metric_record[k] = v.item()
        else:
            metric_record[k] = v
    tr = metric_record.get("threshold_rule")
    if isinstance(tr, str) and tr.startswith("{"):
        try:
            metric_record["threshold_rule"] = ast.literal_eval(tr)
        except Exception:
            pass

    atomic_csv(t / "test_predictions.csv", ep.reset_index(drop=True))
    atomic_json(t / "COMPLETE.json", metric_record)
    atomic_json(
        t / "TRIAL_VERIFICATION.json",
        {
            "status": "RECOVERED_FROM_PINNED_V4_GOOD38_EVIDENCE",
            "model_key": model,
            "seed": int(seed),
            "outer_fold": int(fold_one),
            "evidence_zip_sha256": EXPECTED_EVIDENCE_SHA256,
            "locked_split_sha256": locked_sha,
            "prediction_rows": int(len(ep)),
            "training_performed": False,
            "recovered_at_ns": time.time_ns(),
        },
    )

    verify = validate_good(model, seed, fold_one)
    if not verify.get("valid"):
        raise RuntimeError(
            f"Post-recovery verification failed for {(model, seed, fold_one)}: {verify}"
        )
    recovered.append((model, seed, fold_one))
    print(
        "RECOVERED GOOD TRIAL — NO TRAINING:",
        f"{model} seed={seed} fold={fold_one} | rows={len(ep)}"
    )

remaining = [(x, validate_good(*x)) for x in GOOD38 if not validate_good(*x).get("valid")]
if remaining:
    raise RuntimeError(f"GOOD38 final validation failed: {remaining[:3]}")

print("=" * 78)
print("GOOD38 RECOVERY FINAL: PASS")
print(f"Already valid/untouched: {len(already_valid)}")
print(f"Recovered from V4 evidence, NO TRAINING: {len(recovered)}")
print("All 38 good identities match the existing locked split.")
print("Launching pinned CPU Repair-7 runner; only BAD7 may train.")
print("=" * 78)

raw = urllib.request.urlopen(FINAL_FIXED_URL, timeout=120).read()
nb = json.loads(raw.decode("utf-8"))
cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
if len(cells) != 1:
    raise RuntimeError(f"Expected one code cell in pinned CPU runner, found {len(cells)}")
runner_code = "".join(cells[0]["source"])
compile(runner_code, "DFU_Repair7_CPU_FINAL_FIXED_launcher.py", "exec")
exec(compile(runner_code, "DFU_Repair7_CPU_FINAL_FIXED_launcher.py", "exec"), globals())
